In [1]:
import pickle
from IPython import display as ipd

In [2]:
# !pip3 install -q byted-frechet-audio-distance

In [16]:
import re
import unicodedata

def slugify(value, allow_unicode=False):
    """
    Taken from https://github.com/django/django/blob/master/django/utils/text.py
    Convert to ASCII if 'allow_unicode' is False. Convert spaces or repeated
    dashes to single dashes. Remove characters that aren't alphanumerics,
    underscores, or hyphens. Convert to lowercase. Also strip leading and
    trailing whitespace, dashes, and underscores.
    """
    value = str(value)
    if allow_unicode:
        value = unicodedata.normalize('NFKC', value)
    else:
        value = unicodedata.normalize('NFKD', value).encode('ascii', 'ignore').decode('ascii')
    value = re.sub(r'[^\w\s-]', '', value.lower())
    return re.sub(r'[-\s]+', '-', value).strip('-_')


In [17]:
# from typing import List
# from frechet_audio_distance.models.vggish
# from frechet_audio_distance.models.mulan import MulanModel, covert_mulan_ckpt_to_model
# from frechet_audio_distance.mulan_similarity_score import MulanSimilarityScore

# ckpt_path = "young-mulan-shortform-step=028000-median_rank_1=149-kaggle_merged.pt"

# audio, text = load_musiccaps_data()

# mulan_models = covert_mulan_ckpt_to_model(ckpt_path, device="cuda")

# # model = MulanModel(text_encoder=mulan_models["text_encoder"], music_encoder=mulan_models["music_encoder"])
# # mulan_score = MulanSimilarityScore(model)
# # mulan_score = mulan_score.to("cuda")

# # audio = audio.to("cuda")
# # score = mulan_score(audio, text, batch_size=128)

# # print(f"Average MuLan Similarity Score: {score.mean()}")


In [23]:
from glob import glob
pkls = sorted(glob("/mnt/bn/audio-diffusion/results/*.pkl"), key=lambda e: int(e.split("-")[-2]))

pkls

['/mnt/bn/audio-diffusion/results/generation-0-0.pkl',
 '/mnt/bn/audio-diffusion/results/generation-1-0.pkl']

In [24]:
n = 0
ds = []
for pkl in pkls:
    with open(pkl, "rb") as f:
        obj = pickle.load(f)

    for batch_idx in range(len(obj["audio"])):
        
        prompt = obj["prompt"][batch_idx]
        category = obj["categories"][batch_idx]
        mulan_token_ids = obj["mulan_token_ids"][batch_idx]
        semantic_token_ids = obj["semantic_token_ids"][batch_idx]
        audio = obj["audio"][batch_idx]

        d = {
            "prompt": prompt,
            "category": category,
            "mulan_token_ids": mulan_token_ids,
            "semantic_token_ids": semantic_token_ids,
            "audio": audio,
            "fn": slugify(prompt)[:128]
        }
        ds.append(d)
        
        # print(obj["prompt"])
        # ipd.display(ipd.Audio(obj["audio"][batch_idx], rate=24000))                
        # torchaudio.save(f"{n}. {fn}.mp3", audio, 24000)
        
        

In [25]:
import os
import torchaudio
import pandas as pd

df = pd.DataFrame(ds)
n = 0
for dir_name, group in df.groupby("category"):
    for idx, row in group.iterrows():
        os.makedirs(dir_name, exist_ok=True)
        fn = row["fn"]
        fn = os.path.join(dir_name, f"{n}. {fn}.mp3")
        audio = row["audio"]
        torchaudio.save(fn, audio, 24000)
        n += 1

In [26]:
!zip -r audio.zip *

updating: Audio Generation From Rich Captions/ (stored 0%)
updating: Audio Generation From Rich Captions/0. the-main-soundtrack-of-an-arcade-game-it-is-fast-paced-and-upbeat-with-a-catchy-electric-guitar-riff-the-music-is-repetitive-and.mp3 (deflated 2%)
updating: Audio Generation From Rich Captions/1. a-fusion-of-reggaeton-and-electronic-dance-music-with-a-spacey-otherworldly-sound-induces-the-experience-of-being-lost-in-space-.mp3 (deflated 4%)
updating: Audio Generation From Rich Captions/2. a-rising-synth-is-playing-an-arpeggio-with-a-lot-of-reverb-it-is-backed-by-pads-sub-bass-line-and-soft-drums-this-song-is-full-o.mp3 (deflated 3%)
updating: evaluate.ipynb (deflated 67%)
  adding: Audio Generation From Rich Captions/3. slow-tempo-bass-and-drums-led-reggae-song-sustained-electric-guitar-high-pitched-bongos-with-ringing-tones-vocals-are-relaxed-wi.mp3 (deflated 2%)
  adding: Audio Generation From Rich Captions/5. meditative-song-calming-and-soothing-with-flutes-and-guitars-the-mus